## Setup Cells

In [0]:
%pip install langgraph langchain langchain-openai pydantic

In [0]:
import os

from typing import Literal, List, Dict
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

os.environ["OPENAI_API_KEY"] = dbutils.secrets.get(scope="llm-scope", key="openai-key")

llm = ChatOpenAI(model="gpt-4o", temperature=0)

## Analyser Agent

In [0]:
class IntentFilter(BaseModel):
    column: str
    operator: str
    value: str

class IntentResponse(BaseModel):
    intent_type: Literal["count", "select", "aggregate", "other"] = "select"
    targets: List[str] = Field(default_factory=list)
    filters: List[IntentFilter] = Field(default_factory=list)
    group_by: List[str] = Field(default_factory=list)
    complexity: Literal["low", "medium", "high"] = "low"

class SchemaMapResponse(BaseModel):
    tables: List[str] = Field(default_factory=list)
    join_keys: List[List[str]] = Field(default_factory=list)
    columns: Dict[str, List[str]] = Field(default_factory=dict)

In [0]:
ANALYZER_PROMPT = ChatPromptTemplate.from_messages([
    ("system", """You are a healthcare data query analyzer working with Synthea data.
Parse the user query and return a JSON object with these exact fields:
- intent_type: one of count, select, aggregate, other
- targets: list of ALL entity types involved. Always include related entities 
  (e.g., if query mentions a condition, include both "patients" and "conditions")
- filters: list of objects with column, operator, value
- group_by: list of grouping fields if any
- complexity: low, medium, or high

Return ONLY valid JSON. No explanation, no markdown, no backticks."""),
    ("human", "{query}")
])

def analyzer_node(state: dict) -> dict:
    try:
        chain = ANALYZER_PROMPT | llm
        response = chain.invoke({"query": state["query"]})
        intent = IntentResponse.model_validate_json(response.content)
        return {**state, "intent": intent.model_dump()}
    except Exception as e:
        return {
            **state,
            "intent": None,
            "errors": state.get("errors", []) + [f"Analyzer error: {str(e)}"]
        }

In [0]:
# test_state = {
#     "query": "How many female patients have diabetes?",
#     "errors": []
# }

# result = analyzer_node(test_state)
# print("Intent:", result["intent"])
# print("Errors:", result["errors"])

## Schema Mapper Agent

In [0]:
SYNTHEA_SCHEMA = {
    "tables": {
        "patients":   ["Id", "BIRTHDATE", "DEATHDATE", "GENDER", "RACE", "CITY", "STATE"],
        "conditions": ["START", "STOP", "PATIENT", "ENCOUNTER", "CODE", "DESCRIPTION"],
        "medications":["START", "STOP", "PATIENT", "ENCOUNTER", "CODE", "DESCRIPTION", "REASONCODE"],
        "observations":["DATE", "PATIENT", "ENCOUNTER", "CODE", "DESCRIPTION", "VALUE", "UNITS", "TYPE"],
        "encounters": ["Id", "START", "STOP", "PATIENT", "CODE", "DESCRIPTION", "REASONCODE"],
        "procedures": ["DATE", "PATIENT", "ENCOUNTER", "CODE", "DESCRIPTION"]
    },
    "join_keys": {
        "conditions":  ["patients.Id", "conditions.PATIENT"],
        "medications": ["patients.Id", "medications.PATIENT"],
        "observations":["patients.Id", "observations.PATIENT"],
        "encounters":  ["patients.Id", "encounters.PATIENT"],
        "procedures":  ["patients.Id", "procedures.PATIENT"]
    }
}

In [0]:
SCHEMA_PROMPT = ChatPromptTemplate.from_messages([
    ("system", """You are a healthcare database schema expert working with Synthea data.
Given the user intent, SELECT ONLY the relevant tables and columns needed to answer the query.
Do NOT return the entire schema.

Return a JSON object with:
- tables: list of ONLY the table names needed for this specific query
- join_keys: list of [left_key, right_key] pairs needed (only if joining tables)
- columns: dict of table -> list of ONLY the columns needed for this specific query

Available schema for reference:
{schema_reference}

Return ONLY valid JSON. No explanation, no markdown, no backticks."""),
    ("human", "Intent: {intent}")
])

def schema_mapper_node(state: dict) -> dict:
    try:
        chain = SCHEMA_PROMPT | llm
        response = chain.invoke({
            "intent": str(state["intent"]),
            "schema_reference": str(SYNTHEA_SCHEMA)
        })
        schema = SchemaMapResponse.model_validate_json(response.content)
        return {**state, "schema": schema.model_dump()}
    except Exception as e:
        return {
            **state,
            "schema": None,
            "errors": state.get("errors", []) + [f"Schema mapper error: {str(e)}"]
        }

In [0]:
# test_state = {
#     "query": "How many female patients have diabetes?",
#     "intent": {
#         "intent_type": "count",
#         "targets": ["patients", "conditions"],
#         "filters": [
#             {"column": "gender", "operator": "=", "value": "female"},
#             {"column": "condition", "operator": "=", "value": "diabetes"}
#         ],
#         "group_by": [],
#         "complexity": "low"
#     },
#     "errors": []
# }

# result = schema_mapper_node(test_state)
# print("Schema:", result["schema"])
# print("Errors:", result["errors"])

## SQL Generator Agent

In [0]:
def sample_column_values(table: str, column: str, n: int = 5) -> list:
    try:
        rows = spark.sql(f"SELECT DISTINCT {column} FROM {table} LIMIT {n}").collect()
        return [row[0] for row in rows]
    except:
        return []

In [0]:
SQL_GEN_PROMPT = ChatPromptTemplate.from_messages([
    ("system", """You are a Spark SQL expert working with Synthea healthcare data.
Generate a valid Spark SQL query based on the intent and schema provided.

Rules:
- Use table aliases: p=patients, c=conditions, m=medications, o=observations, e=encounters
- Use ONLY the actual sampled column values provided for filters (not assumed values)
- For partial text matches use LOWER(column) LIKE LOWER('%keyword%')
- Always qualify column names with table alias
- Return ONLY the SQL query. No explanation, no markdown, no backticks."""),
    ("human", """Intent: {intent}
Schema: {schema}
Sampled column values: {sampled_values}
Previous errors (if any): {errors}""")
])

def sql_generator_node(state: dict) -> dict:
    try:
        schema = state["schema"]

        # Sample actual values for relevant columns
        sampled = {}
        for table in schema.get("tables", []):
            for column in schema.get("columns", {}).get(table, []):
                values = sample_column_values(table, column)
                if values:
                    sampled[f"{table}.{column}"] = values

        # Use higher temperature on retries
        temp = 0.0 if state.get("retry_count", 0) == 0 else 0.3
        gen_llm = ChatOpenAI(model="gpt-4o", temperature=temp)

        chain = SQL_GEN_PROMPT | gen_llm
        response = chain.invoke({
            "intent": str(state["intent"]),
            "schema": str(schema),
            "sampled_values": str(sampled),
            "errors": state.get("errors", [])
        })

        return {**state, "sql": response.content.strip()}

    except Exception as e:
        return {
            **state,
            "sql": None,
            "errors": state.get("errors", []) + [f"SQL generator error: {str(e)}"]
        }

In [0]:
# test_state = {
#     "query": "How many female patients have diabetes?",
#     "intent": {
#         "intent_type": "count",
#         "targets": ["patients", "conditions"],
#         "filters": [
#             {"column": "gender", "operator": "=", "value": "female"},
#             {"column": "condition", "operator": "=", "value": "diabetes"}
#         ],
#         "group_by": [],
#         "complexity": "low"
#     },
#     "schema": {
#         "tables": ["patients", "conditions"],
#         "join_keys": [["patients.Id", "conditions.PATIENT"]],
#         "columns": {
#             "patients": ["Id", "GENDER"],
#             "conditions": ["PATIENT", "DESCRIPTION"]
#         }
#     },
#     "errors": [],
#     "retry_count": 0
# }

# result = sql_generator_node(test_state)
# print("SQL:", result["sql"])
# print("Errors:", result["errors"])